<a href="https://colab.research.google.com/github/weaamasad99/CloudComputingWolf/blob/main/TUT6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install paho-mqtt


In [ ]:
import paho.mqtt.client as mqtt
import json

In [ ]:
# Callback function for when a message is received
def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode())

        # Extract values with lowercase field names
        temperature = data.get("temperature", "N/A")
        humidity = data.get("humidity", "N/A")
        soil= data.get("soil", "N/A")
        print(f"Temperature: {temperature}°C, Humidity: {humidity}%")

    except json.JSONDecodeError:
        print("Received invalid JSON data")
        # MQTT setup
broker = "io.adafruit.com"
username = "braude2"   #insert Adafruit user name
aio_key = "key" #insert key from Adafruit
topic = f"{username}/feeds/json"
client = mqtt.Client()
client.username_pw_set(username,aio_key)
client.on_message = on_message

client.connect(broker, 1883, 60)
client.subscribe(topic)

print(f"Subscribed to MQTT topic: {topic}")
client.loop_forever()


In [ ]:
import requests

USERNAME = "braude2"   #insert Adafruit user name
AIO_KEY = "key" #insert key from Adafruit
FEED = "json"

url = f"https://io.adafruit.com/api/v2/{USERNAME}/feeds/{FEED}/data"
headers = {"X-AIO-Key": AIO_KEY}

response = requests.get(url, headers=headers)
data = response.json()

for item in data[:50]:  # חמשת הנתונים האחרונים
    print(f"Value: {item['value']}, Time: {item['created_at']}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for plotting
temps = []
times = []

# Iterate through the data, ensuring we only take up to 100 items
for item in data[:100]:
    try:
        # The 'value' field is a JSON string, so we need to parse it
        value_dict = json.loads(item['value'])
        temperature = value_dict.get('temperature')

        if temperature is not None:
            temps.append(temperature)
            times.append(item['created_at'])
    except json.JSONDecodeError:
        print(f"Could not decode JSON for item: {item['value']}")

# Create a DataFrame for easier plotting
df_temp = pd.DataFrame({
    'Time': pd.to_datetime(times),
    'Temperature': temps
}).sort_values(by='Time', ascending=True) # Sort by time to ensure correct line plot

print(f"Displaying {len(df_temp)} temperature readings.")
display(df_temp.head())

In [ ]:
# Plot the temperatures
plt.figure(figsize=(12, 6))
sns.lineplot(x='Time', y='Temperature', data=df_temp)
plt.title('Last 100 Temperature Readings Over Time')
plt.xlabel('Time')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()